# MSI 5102 - Handwritten Digit Recognition (MNIST)

This project implements and compares multiple machine learning models for classifying handwritten digits from the MNIST dataset.

**Models implemented:**
1. k-Nearest Neighbors (KNN)
2. Logistic Regression
3. Neural Network (MLP & CNN) — Bonus

**Additional features:**
- Dimensionality reduction & visualization (PCA, t-SNE)
- Hyperparameter optimization for each model
- Compatible with MacBook Apple Silicon (MPS) and Kaggle Free GPU (CUDA)

## 1. Environment Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms

# Device detection: MPS (MacBook) > CUDA (Kaggle) > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS (MacBook)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"Device: {device}")
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

In [ ]:
# Load MNIST dataset
print("Loading MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

X, y = mnist.data.astype('float32'), mnist.target.astype('int')

# Normalize pixel values to [0, 1]
X = X / 255.0

# Split into train and test sets (60000 / 10000)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Image shape:  28x28 = {X_train.shape[1]} features")
print(f"Classes:      {np.unique(y_train)}")

## 2. Data Visualization

In [ ]:
# Display sample images for each digit class (5 samples per digit)
fig, axes = plt.subplots(10, 5, figsize=(8, 16))
fig.suptitle("Sample Images for Each Digit (0-9)", fontsize=16, y=1.01)

for digit in range(10):
    indices = np.where(y_train == digit)[0][:5]
    for j, idx in enumerate(indices):
        axes[digit, j].imshow(X_train[idx].reshape(28, 28), cmap='gray')
        axes[digit, j].axis('off')
        if j == 0:
            axes[digit, j].set_title(f"Digit {digit}", fontsize=12, loc='left')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title in zip(axes, [y_train, y_test], ['Training Set', 'Test Set']):
    unique, counts = np.unique(data, return_counts=True)
    ax.bar(unique, counts, color=sns.color_palette("husl", 10))
    ax.set_xlabel("Digit")
    ax.set_ylabel("Count")
    ax.set_title(f"Class Distribution - {title}")
    ax.set_xticks(range(10))

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Model Implementation

We will store results for comparison later.

In [ ]:
# Dictionary to store results for all models
results = {}

def evaluate_model(name, y_true, y_pred, train_time, predict_time):
    """Evaluate a model and store results."""
    acc = accuracy_score(y_true, y_pred)
    results[name] = {
        'accuracy': acc,
        'train_time': train_time,
        'predict_time': predict_time,
        'y_pred': y_pred
    }
    print(f"\n{'='*50}")
    print(f"{name} - Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"Training time: {train_time:.2f}s | Prediction time: {predict_time:.2f}s")
    print(f"{'='*50}")
    print(f"\nClassification Report:\n")
    print(classification_report(y_true, y_pred))

### 3(a) k-Nearest Neighbors (KNN)

We first apply PCA for dimensionality reduction to speed up KNN, then search for the best combination of:
- Number of PCA components (30, 50, 100)
- Distance metric (euclidean, manhattan)
- k values (1, 3, 5, 7, 9)
- Voting weights (uniform, distance)

In [ ]:
# Step 1: Find optimal PCA components for KNN using a subset for speed
print("Searching for best PCA dimensions for KNN...")

# Use a subset for cross-validation to save time
subset_size = 10000
indices = np.random.RandomState(42).choice(len(X_train), subset_size, replace=False)
X_sub, y_sub = X_train[indices], y_train[indices]

pca_results = {}
for n_comp in [30, 50, 100]:
    pca_temp = PCA(n_components=n_comp, random_state=42)
    X_sub_pca = pca_temp.fit_transform(X_sub)
    
    knn_temp = KNeighborsClassifier(n_neighbors=3, weights='distance', metric='euclidean')
    scores = cross_val_score(knn_temp, X_sub_pca, y_sub, cv=3, scoring='accuracy')
    pca_results[n_comp] = scores.mean()
    print(f"  PCA({n_comp}): CV accuracy = {scores.mean():.4f} ± {scores.std():.4f}")

best_n_comp = max(pca_results, key=pca_results.get)
print(f"\nBest PCA components: {best_n_comp}")

# Apply PCA with best components on full data
pca_knn = PCA(n_components=best_n_comp, random_state=42)
X_train_pca = pca_knn.fit_transform(X_train)
X_test_pca = pca_knn.transform(X_test)
print(f"Variance explained: {pca_knn.explained_variance_ratio_.sum():.4f}")

In [ ]:
# Step 2: Search for best k, metric, and weights
print("Searching for best KNN hyperparameters...")

k_values = [1, 3, 5, 7, 9]
param_grid = {
    'n_neighbors': k_values,
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# Use subset with PCA-reduced data for faster grid search
X_sub_pca = pca_knn.transform(X_sub)
grid_search = GridSearchCV(
    KNeighborsClassifier(), param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search.fit(X_sub_pca, y_sub)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

# Plot k vs accuracy for different configurations
cv_results = pd.DataFrame(grid_search.cv_results_)
fig, ax = plt.subplots(figsize=(10, 6))

for metric in ['euclidean', 'manhattan']:
    for weight in ['uniform', 'distance']:
        mask = ((cv_results['param_metric'] == metric) & 
                (cv_results['param_weights'] == weight))
        subset = cv_results[mask].sort_values('param_n_neighbors')
        label = f"{metric}, {weight}"
        ax.plot(subset['param_n_neighbors'], subset['mean_test_score'], 
                marker='o', label=label)

ax.set_xlabel("k (Number of Neighbors)")
ax.set_ylabel("Cross-Validation Accuracy")
ax.set_title("KNN: k vs Accuracy for Different Configurations")
ax.legend()
ax.set_xticks(k_values)
plt.tight_layout()
plt.savefig('knn_k_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 3: Train final KNN with best parameters on full training set
best_params = grid_search.best_params_
print(f"Training KNN with best parameters: {best_params}")

knn_model = KNeighborsClassifier(**best_params)

t_start = time.time()
knn_model.fit(X_train_pca, y_train)
knn_train_time = time.time() - t_start

t_start = time.time()
knn_pred = knn_model.predict(X_test_pca)
knn_predict_time = time.time() - t_start

evaluate_model("KNN (Optimized)", y_test, knn_pred, knn_train_time, knn_predict_time)

### 3(b) Logistic Regression

We use multinomial logistic regression with hyperparameter tuning over regularization strength (C) and solver.

In [ ]:
# Logistic Regression with GridSearchCV
print("Searching for best Logistic Regression hyperparameters...")

lr_param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'solver': ['lbfgs', 'saga'],
    'penalty': ['l2']
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    lr_param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)

# Use PCA-reduced data for faster training
lr_grid.fit(X_sub_pca, y_sub)

print(f"\nBest parameters: {lr_grid.best_params_}")
print(f"Best CV accuracy: {lr_grid.best_score_:.4f}")

# Plot C vs accuracy
lr_cv_results = pd.DataFrame(lr_grid.cv_results_)
fig, ax = plt.subplots(figsize=(8, 5))

for solver in ['lbfgs', 'saga']:
    mask = lr_cv_results['param_solver'] == solver
    subset = lr_cv_results[mask].sort_values('param_C')
    ax.plot(subset['param_C'], subset['mean_test_score'], marker='o', label=f"solver={solver}")
    ax.fill_between(subset['param_C'],
                     subset['mean_test_score'] - subset['std_test_score'],
                     subset['mean_test_score'] + subset['std_test_score'], alpha=0.2)

ax.set_xscale('log')
ax.set_xlabel("Regularization Strength (C)")
ax.set_ylabel("Cross-Validation Accuracy")
ax.set_title("Logistic Regression: C vs Accuracy")
ax.legend()
plt.tight_layout()
plt.savefig('lr_c_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Train final Logistic Regression with best params on full training set
best_lr_params = lr_grid.best_params_
print(f"Training Logistic Regression with best parameters: {best_lr_params}")

lr_model = LogisticRegression(
    max_iter=2000, random_state=42, **best_lr_params
)

t_start = time.time()
lr_model.fit(X_train_pca, y_train)
lr_train_time = time.time() - t_start

t_start = time.time()
lr_pred = lr_model.predict(X_test_pca)
lr_predict_time = time.time() - t_start

evaluate_model("Logistic Regression (Optimized)", y_test, lr_pred, lr_train_time, lr_predict_time)

### 3(c) Neural Network (Bonus)

We implement two architectures:
1. **MLP** — Multi-Layer Perceptron with BatchNorm and Dropout
2. **CNN** — Convolutional Neural Network for higher accuracy

Both use data augmentation, learning rate scheduling, and early stopping.

In [ ]:
# Prepare PyTorch datasets
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

# Split training into train/val for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_tensor, y_train_tensor, test_size=0.1, random_state=42, stratify=y_train
)

# Data augmentation transform for CNN (rotation ±10°, translation ±10%)
cnn_augment = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
])

train_dataset = TensorDataset(X_tr, y_tr)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print(f"\nCNN Data Augmentation:")
print(f"  - RandomRotation: ±10 degrees")
print(f"  - RandomAffine translation: ±10% (±2.8 pixels)")
print(f"MLP Data Augmentation:")
print(f"  - Gaussian noise: std=0.05")

In [ ]:
# Define MLP model
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(784, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        return self.network(x)

# Define CNN model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 28x28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 14x14
            nn.Dropout(0.25),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 14x14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 7x7
            nn.Dropout(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )
    
    def forward(self, x):
        x = x.view(-1, 1, 28, 28)
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

print("MLP architecture:")
print(MLP())
print(f"\nCNN architecture:")
print(CNN())

In [ ]:
def train_nn(model, train_loader, val_loader, epochs=20, lr=0.001, model_name="NN"):
    """Train a neural network with early stopping and LR scheduling."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0
    best_model_state = None
    patience_counter = 0
    patience = 5
    
    t_start = time.time()
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # Data augmentation
            if model_name == "MLP":
                # MLP: add Gaussian noise (std=0.05)
                X_batch = X_batch + torch.randn_like(X_batch) * 0.05
                X_batch = X_batch.clamp(0, 1)
            elif model_name == "CNN":
                # CNN: rotation ±10°, translation ±10%
                X_img = X_batch.view(-1, 1, 28, 28)
                X_img = cnn_augment(X_img)
                X_batch = X_img.view(-1, 784)
            
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * X_batch.size(0)
            train_correct += (output.argmax(1) == y_batch).sum().item()
            train_total += X_batch.size(0)
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                val_correct += (output.argmax(1) == y_batch).sum().item()
                val_total += X_batch.size(0)
        
        train_loss /= train_total
        val_loss /= val_total
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f"Epoch {epoch+1:2d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"LR: {current_lr:.6f}")
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    train_time = time.time() - t_start
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    return model, history, train_time

In [ ]:
def predict_nn(model, test_loader):
    """Get predictions from a neural network."""
    model.eval()
    all_preds = []
    t_start = time.time()
    with torch.no_grad():
        for X_batch, _ in test_loader:
            X_batch = X_batch.to(device)
            output = model(X_batch)
            all_preds.append(output.argmax(1).cpu())
    predict_time = time.time() - t_start
    return torch.cat(all_preds).numpy(), predict_time

In [ ]:
# Train MLP
print("=" * 60)
print("Training MLP...")
print("=" * 60)

mlp_model = MLP()
mlp_model, mlp_history, mlp_train_time = train_nn(
    mlp_model, train_loader, val_loader, epochs=20, lr=0.001, model_name="MLP"
)

mlp_pred, mlp_predict_time = predict_nn(mlp_model, test_loader)
evaluate_model("MLP (Neural Network)", y_test, mlp_pred, mlp_train_time, mlp_predict_time)

In [ ]:
# Train CNN
print("=" * 60)
print("Training CNN...")
print("=" * 60)

cnn_model = CNN()
cnn_model, cnn_history, cnn_train_time = train_nn(
    cnn_model, train_loader, val_loader, epochs=20, lr=0.001, model_name="CNN"
)

cnn_pred, cnn_predict_time = predict_nn(cnn_model, test_loader)
evaluate_model("CNN (Neural Network)", y_test, cnn_pred, cnn_train_time, cnn_predict_time)

In [ ]:
# Plot training curves for MLP and CNN
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (history, name) in enumerate([(mlp_history, "MLP"), (cnn_history, "CNN")]):
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0, idx].plot(epochs_range, history['train_loss'], label='Train Loss')
    axes[0, idx].plot(epochs_range, history['val_loss'], label='Val Loss')
    axes[0, idx].set_xlabel('Epoch')
    axes[0, idx].set_ylabel('Loss')
    axes[0, idx].set_title(f'{name} - Loss Curve')
    axes[0, idx].legend()
    
    # Accuracy
    axes[1, idx].plot(epochs_range, history['train_acc'], label='Train Acc')
    axes[1, idx].plot(epochs_range, history['val_acc'], label='Val Acc')
    axes[1, idx].set_xlabel('Epoch')
    axes[1, idx].set_ylabel('Accuracy')
    axes[1, idx].set_title(f'{name} - Accuracy Curve')
    axes[1, idx].legend()

plt.tight_layout()
plt.savefig('nn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Evaluation & Comparison

In [ ]:
# Confusion matrices for all models
model_names = list(results.keys())
n_models = len(model_names)

fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, name in zip(axes, model_names):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=range(10), yticklabels=range(10))
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{name}\nAccuracy: {results[name]["accuracy"]:.4f}')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary comparison table
comparison_df = pd.DataFrame({
    'Model': model_names,
    'Accuracy (%)': [results[n]['accuracy'] * 100 for n in model_names],
    'Train Time (s)': [results[n]['train_time'] for n in model_names],
    'Predict Time (s)': [results[n]['predict_time'] for n in model_names],
}).round(4)

comparison_df = comparison_df.sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)
print("=" * 70)
print("MODEL COMPARISON SUMMARY")
print("=" * 70)
print(comparison_df.to_string(index=False))

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Accuracy
axes[0].barh(comparison_df['Model'], comparison_df['Accuracy (%)'], color=sns.color_palette("husl", n_models))
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Test Accuracy Comparison')
axes[0].set_xlim(85, 100)

# Train time
axes[1].barh(comparison_df['Model'], comparison_df['Train Time (s)'], color=sns.color_palette("husl", n_models))
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Training Time Comparison')

# Predict time
axes[2].barh(comparison_df['Model'], comparison_df['Predict Time (s)'], color=sns.color_palette("husl", n_models))
axes[2].set_xlabel('Time (s)')
axes[2].set_title('Prediction Time Comparison')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Discussion: Strengths & Weaknesses

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **KNN** | Simple, no training phase, intuitive, non-parametric | Slow at prediction time (O(n) per query), sensitive to high dimensions, large memory footprint |
| **Logistic Regression** | Fast training & prediction, interpretable weights, good baseline | Linear decision boundary limits accuracy on complex patterns, may underfit |
| **MLP** | Captures non-linear patterns, flexible architecture, fast prediction | Requires hyperparameter tuning, no spatial awareness of image structure |
| **CNN** | Best accuracy, leverages spatial structure of images, translation invariant | Longest training time, more parameters, requires GPU for efficient training |

**Key observations:**
- CNN achieves the highest accuracy by exploiting the 2D spatial structure of digit images through convolutional filters.
- KNN with PCA dimensionality reduction and distance-weighted voting achieves surprisingly good results with minimal complexity.
- Logistic Regression, despite being a linear model, performs reasonably well due to MNIST being a relatively "easy" dataset.
- The accuracy-complexity tradeoff is clear: CNN > MLP > KNN > LR in accuracy, but the reverse in simplicity.

## 5. Dimensionality Reduction & Visualization

In [ ]:
# Use a subset for visualization (t-SNE is computationally expensive)
viz_size = 10000
viz_indices = np.random.RandomState(42).choice(len(X_train), viz_size, replace=False)
X_viz = X_train[viz_indices]
y_viz = y_train[viz_indices]

# PCA to 2D
print("Applying PCA...")
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_viz)
print(f"PCA explained variance ratio: {pca_2d.explained_variance_ratio_}")
print(f"Total variance explained: {pca_2d.explained_variance_ratio_.sum():.4f}")

# t-SNE to 2D (use PCA to 50D first for speed)
print("\nApplying t-SNE (this may take a few minutes)...")
pca_50 = PCA(n_components=50, random_state=42)
X_pca_50 = pca_50.fit_transform(X_viz)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000, learning_rate='auto', init='pca')
X_tsne_2d = tsne.fit_transform(X_pca_50)
print("t-SNE complete.")

In [ ]:
# Plot PCA and t-SNE side by side
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

colors = plt.cm.tab10(np.linspace(0, 1, 10))

for ax, X_2d, title in zip(axes, [X_pca_2d, X_tsne_2d], ['PCA', 't-SNE']):
    for digit in range(10):
        mask = y_viz == digit
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                   c=[colors[digit]], label=str(digit), 
                   alpha=0.5, s=5)
    ax.set_title(f'{title} Visualization of MNIST Digits', fontsize=14)
    ax.set_xlabel(f'{title} Component 1')
    ax.set_ylabel(f'{title} Component 2')
    ax.legend(title='Digit', markerscale=5, loc='best')

plt.tight_layout()
plt.savefig('dimensionality_reduction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PCA variance explained analysis
pca_full = PCA(n_components=50, random_state=42)
pca_full.fit(X_viz)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual variance
axes[0].bar(range(1, 51), pca_full.explained_variance_ratio_, alpha=0.8)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA: Variance Explained by Each Component')

# Cumulative variance
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, 51), cumulative_var, marker='o', markersize=3)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
n_95 = np.argmax(cumulative_var >= 0.95) + 1
axes[1].axvline(x=n_95, color='g', linestyle='--', label=f'{n_95} components for 95%')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('PCA: Cumulative Variance Explained')
axes[1].legend()

plt.tight_layout()
plt.savefig('pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Components needed for 95% variance: {n_95}")

### Interpretation of Clustering Patterns

**PCA Visualization:**
- PCA preserves global structure but clusters overlap significantly in 2D, because the first two principal components only capture a limited portion of total variance.
- Despite overlap, some digits (e.g., 0 and 1) are relatively well separated, while others (e.g., 4 and 9, 3 and 5) show significant overlap, indicating visual similarity.

**t-SNE Visualization:**
- t-SNE produces much clearer, well-separated clusters for each digit class.
- It reveals the local neighborhood structure: similar-looking digits are grouped tightly together.
- Digits that are commonly confused (e.g., 4/9, 3/5, 7/1) may have overlapping regions in the t-SNE plot, which aligns with the confusion matrix results.
- t-SNE is non-linear and better at preserving local structure, making it superior for visualization of high-dimensional data clusters.

**PCA vs t-SNE:**
- PCA is a linear method that maximizes variance — fast but limited in revealing non-linear structure.
- t-SNE is non-linear and focuses on preserving local similarities — much better for visualization but computationally expensive and non-deterministic.

## 6. Optimization Summary

This section summarizes the optimization techniques applied to each model and their impact.

In [ ]:
# Final summary of all optimization techniques
print("=" * 70)
print("OPTIMIZATION TECHNIQUES SUMMARY")
print("=" * 70)

optimization_summary = pd.DataFrame({
    'Model': ['KNN', 'KNN', 'KNN', 
              'Logistic Regression', 'Logistic Regression',
              'MLP', 'MLP', 'MLP', 'MLP',
              'CNN', 'CNN', 'CNN', 'CNN', 'CNN'],
    'Optimization': [
        'PCA dimensionality reduction',
        'Distance-weighted voting',
        'Grid search (k, metric)',
        'Grid search (C, solver)',
        'L2 regularization tuning',
        'BatchNorm + Dropout',
        'Random noise augmentation',
        'Learning rate scheduling',
        'Early stopping',
        'Deeper conv architecture',
        'BatchNorm + Dropout',
        'Learning rate scheduling',
        'Early stopping',
        'Weight decay (L2 reg.)'
    ],
    'Purpose': [
        'Reduce computation, remove noise',
        'Closer neighbors contribute more',
        'Find optimal hyperparameters',
        'Find optimal regularization',
        'Prevent overfitting',
        'Regularization & faster convergence',
        'Improve generalization',
        'Fine-tune learning rate',
        'Prevent overfitting',
        'Better feature extraction',
        'Regularization & faster convergence',
        'Fine-tune learning rate',
        'Prevent overfitting',
        'Weight regularization'
    ]
})

print(optimization_summary.to_string(index=False))

print(f"\n{'=' * 70}")
print("FINAL MODEL PERFORMANCE")
print("=" * 70)
print(comparison_df.to_string(index=False))
print(f"\nBest model: {comparison_df.iloc[0]['Model']} with {comparison_df.iloc[0]['Accuracy (%)']:.2f}% accuracy")